# Final Review Project: RAG-Based Customer Support Assistant
**Author:** AntonyJ  
**Course:** CSE (8th Semester)  
**Objective:** Design and build a reliable AI Support Agent using LangGraph and Retrieval-Augmented Generation (RAG).

### 📌 System Overview
This system is designed to provide accurate answers to customer queries by retrieving information from a specific company knowledge base (PDF). To ensure safety and accuracy, the system includes:
1. **RAG Pipeline:** Context-aware responses based on internal manuals.
2. **Graph-Based Logic:** Managed workflow using LangGraph.
3. **Human-in-the-Loop (HITL):** Automatic escalation to a human supervisor for high-risk or complex queries.

###Step 1: Install Dependencies

In [6]:
!pip install -q langchain-groq langgraph chromadb pypdf sentence-transformers langchain-community

##Step 2: Setup Environments & Imports

## 🏗️ High-Level Design (HLD)

### 1. Architecture Components
* **Document Ingestion:** Processes `data.pdf` into searchable chunks.
* **Vector Database (ChromaDB):** Stores text embeddings for fast semantic retrieval.
* **Orchestration Layer (LangGraph):** Controls the decision-making flow between searching and answering.
* **LLM Layer (Groq/Llama 3.1):** Generates natural language responses based on retrieved context.

### 2. System Workflow
1. **Input:** User asks a support question.
2. **Retrieval:** The system searches the vector store for the top 2 most relevant text chunks.
3. **Reasoning:** The LLM analyzes the chunks.
4. **Decision:** - If the info is found: Answer the user.
    - If the info is missing OR danger keywords (fire, legal) are detected: Escalate to **HITL**.

In [7]:
import os
from typing import TypedDict
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langgraph.graph import StateGraph, END

# Disable symlink warnings for Colab's file system
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

# Set your Groq API Key
os.environ["GROQ_API_KEY"] = "Paste_groq_api_here"

print("Setup Complete.")

Setup Complete.


##Step 3: Document Processing (The RAG Pipeline)

## ⚙️ Low-Level Design (LLD)

### 1. Data Processing Details
* **Chunking Strategy:** `RecursiveCharacterTextSplitter` with `chunk_size=600` and `overlap=100`. This ensures context is preserved across split boundaries.
* **Embeddings:** `all-MiniLM-L6-v2` via HuggingFace. This is a 384-dimensional dense vector model optimized for speed and semantic similarity.

### 2. Graph State Definition
We use a `TypedDict` called `AgentState` to pass data between nodes:
- `question`: The original user query.
- `context`: The retrieved text from the PDF.
- `answer`: The final generated response.
- `needs_human`: A boolean flag for escalation.

### 3. Conditional Routing Logic
The system uses a hard-coded "Safety Router." If the query contains keywords like **"fire", "lawyer", or "smoke"**, the `needs_human` flag is set to `True`, bypassing the standard AI response to ensure human oversight.

In [ ]:
print("--- INITIALIZING KNOWLEDGE BASE ---")
try:
    # Ensure data.pdf is uploaded to the sidebar!
    loader = PyPDFLoader("data.pdf")
    docs = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100)
    splits = text_splitter.split_documents(docs)

    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
    print("--- PDF LOADED AND INDEXED SUCCESSFULLY ---")
except Exception as e:
    print(f"Error: {e}. Did you forget to upload 'data.pdf' to the sidebar?")

##Step 4: Define Graph Logic (State & Nodes)

## 🛠️ Technical Challenges & Solutions

| Challenge | Solution |
| :--- | :--- |
| **Model Deprecation** | Migrated from `llama3-8b` to `llama-3.1-8b-instant` via Groq to maintain production stability. |
| **Method Updates** | Updated legacy `get_relevant_documents` calls to the modern LangChain `.invoke()` method. |
| **Safety Concerns** | Implemented a Keyword-Based Trigger within the Assistant Node to force Human-in-the-Loop escalation for liability issues. |
| **Environment Sync** | Used `os.environ` to manage API keys and disable symlink warnings on Windows/Colab environments. |

In [9]:
class AgentState(TypedDict):
    question: str
    context: str
    answer: str
    needs_human: bool

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

def retrieve_node(state: AgentState):
    print("[Node: Retriever] Searching document...")
    question = state["question"]
    docs = retriever.invoke(question)
    context = "\n\n".join([d.page_content for d in docs])
    return {"context": context}

def assistant_node(state: AgentState):
    print("[Node: Assistant] Analyzing context...")
    question = state["question"].lower()
    context = state["context"]

    danger_keywords = ["fire", "smoke", "legal", "lawyer", "sue", "explosion"]

    if not context.strip() or any(word in question for word in danger_keywords):
        return {"needs_human": True, "answer": "I cannot handle this request safely."}

    prompt = f"Using this context: {context}, answer: {state['question']}"
    response = llm.invoke(prompt)
    return {"answer": response.content, "needs_human": False}

# Build Workflow
workflow = StateGraph(AgentState)
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("assistant", assistant_node)
workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "assistant")
workflow.add_edge("assistant", END)

app = workflow.compile()
print("Workflow Graph Compiled.")

Workflow Graph Compiled.


##Step 5: The Interactive Chat Bot (HITL Interface)

In [10]:
def main():
    print("==============================================")
    print("   SWIFTTECH AI CUSTOMER SUPPORT SYSTEM       ")
    print("==============================================\n")

    while True:
        user_input = input("Customer: ")
        if user_input.lower() in ["exit", "quit", "bye"]:
            break

        inputs = {
            "question": user_input,
            "context": "",
            "answer": "",
            "needs_human": False
        }

        final_output = None
        for output in app.stream(inputs):
            for key, value in output.items():
                final_output = value

        if final_output.get("needs_human"):
            print("\n[!] SYSTEM ALERT: High-priority issue detected.")
            print("[!] STATUS: Escalating to Human Supervisor...")
            human_response = input("\n[HUMAN SUPERVISOR OVERRIDE]: ")
            print(f"\nFINAL RESPONSE SENT TO CUSTOMER: {human_response}\n")
        else:
            print(f"\nAI ASSISTANT: {final_output.get('answer')}\n")

        print("-" * 40)

if __name__ == "__main__":
    main()

   SWIFTTECH AI CUSTOMER SUPPORT SYSTEM       

Customer: What is the warranty period for a smartphone?
[Node: Retriever] Searching document...
[Node: Assistant] Analyzing context...

AI ASSISTANT: According to the SwiftTech Electronics Support Manual, Version 2026.1, the warranty period for a flagship smartphone is 24 months (2 years).

----------------------------------------
Customer: Can I return an item after 45 days?
[Node: Retriever] Searching document...
[Node: Assistant] Analyzing context...

AI ASSISTANT: According to the Return and Refund Process, customers can return products within 30 days of purchase for a full refund if the item is in its original packaging. Since 45 days is beyond the 30-day window, the answer is no, you cannot return an item after 45 days.

----------------------------------------
Customer: Is there a fee for returning opened electronics?
[Node: Retriever] Searching document...
[Node: Assistant] Analyzing context...

AI ASSISTANT: Yes, there is a fee f